# Adding commands to the earthkit CLI

earthkit-utils provides the `earthkit` command line tool. It does not do much by itself: each earthkit package
adds its own commands to it, e.g. earthkit-data adds `earthkit ls` and earthkit-geo adds `earthkit regrid`. The
commands are [click](https://click.palletsprojects.com) commands registered on the shared `earthkit` group, and
the shared arguments and options of `earthkit.cli.standard_args` give them a consistent interface for reading
and writing data with earthkit-data.

This tutorial builds a few commands step by step:

1. a first command, to see how commands are registered,
2. a command reading data from an earthkit-data *source*,
3. a command writing data to an earthkit-data *target*,
4. a command with several sources, given as options,
5. a command using the shared `--index` option,

and then shows how to ship the commands in a package. See [Command line interface standards](../concepts/cli.rst) for how the
command line interface works.

<div class="alert alert-info">

**Note:** In a package, the commands live in a module of the `earthkit.cli` namespace package and are run from
a shell, e.g. `earthkit hello`. In this notebook we register them directly, and run them with click's
`CliRunner` through a small `run` helper, which takes the same command line you would type after `earthkit`.

</div>

## Setup

The commands need [click](https://click.palletsprojects.com), installed with earthkit-utils, and earthkit-data
to read and write data. The example data is created with Xarray in a temporary directory.

In [1]:
import os
import shlex
import tempfile

import click
import numpy as np
import pandas as pd
import xarray as xr
from click.testing import CliRunner

from earthkit.cli.main import earthkit
from earthkit.cli.standard_args import (
    SOURCE_HELP,
    TARGET_HELP,
    add_options,
    index_option,
    profile_option,
    source_options,
    split_csv,
    target_options,
)

os.chdir(tempfile.mkdtemp())


def run(command_line):
    # Run 'earthkit <command_line>' and print its output, as a shell would
    result = CliRunner().invoke(earthkit, shlex.split(command_line), prog_name="earthkit", terminal_width=100)
    print(result.output, end="")
    if result.exception and not isinstance(result.exception, SystemExit):
        raise result.exception

In [2]:
time = pd.date_range("2020-01-01", periods=48, freq="6h")
rng = np.random.default_rng(0)
ds = xr.Dataset(
    {
        "t2m": (("time", "latitude", "longitude"), 280 + rng.normal(size=(48, 3, 4)), {"units": "K"}),
        "tp": (("time", "latitude", "longitude"), rng.random((48, 3, 4)), {"units": "m"}),
    },
    coords={"time": time, "latitude": [50.0, 45.0, 40.0], "longitude": [0.0, 5.0, 10.0, 15.0]},
)
ds.to_netcdf("input.nc")

# The same data split in two files, to show reading several sources
ds.isel(time=slice(0, 24)).to_netcdf("part1.nc")
ds.isel(time=slice(24, None)).to_netcdf("part2.nc")

## 1. A first command

A command is a function decorated with `@earthkit.command()`, where `earthkit` is the shared group of
`earthkit.cli.main`. Any click argument or option can be added to it, and its docstring is the help shown by
`earthkit hello -h`:

In [3]:
@earthkit.command()
@click.option("--name", default="world", show_default=True, help="Who to greet.")
def hello(name):
    """Print a greeting."""
    click.echo(f"Hello, {name}!")


run("hello -h")

Usage: earthkit hello [OPTIONS]

  Print a greeting.

Options:
  --name TEXT  Who to greet.  [default: world]
  -h, --help   Show this message and exit.


In [4]:
run("hello --name earthkit")

Hello, earthkit!


The name of the command is the name of the function, with underscores replaced by dashes, e.g. a function
`time_mean` gives the command `time-mean`. Use `@earthkit.command(name="...")` to choose another name.

Following the [earthkit CLI standards](../concepts/cli.rst#short-flags), the options of a command are long-only, like `--name`: short flags are reserved for the common arguments `-s`, `-t`, `-p` and `-i` used below.

## 2. Reading data

`source_options()` adds an earthkit-data source to a command. With `positional=True`, it adds the `SOURCE`
positional argument; otherwise it adds the `--source`/`-s` option (see [section 4](#4.-Several-sources)). The
command receives the object returned by `earthkit.data.from_source()`, so it does not need to open the data
itself.

`add_options()` applies a list of arguments and options in the order they are listed. Click arguments have no
help of their own, so describe `SOURCE` in the help of the command with `SOURCE_HELP`. The help is given with
`help=` here rather than as a docstring, as it is an f-string.

`profile_option` adds the shared `--profile`/`-p` option, which chooses the earthkit-data profile used when
converting the data to Xarray. It is passed to the command as a list of profiles, or `None` if not given.

In [5]:
@earthkit.command(
    help=f"""Summarise the variables of SOURCE.

SOURCE: {SOURCE_HELP}
"""
)
@add_options([source_options(positional=True), profile_option])
def summary(source, profile):
    ds = source.to_xarray(**({"profile": profile} if profile else {}))
    for name, var in ds.data_vars.items():
        click.echo(f"{name}: {dict(var.sizes)}, mean {float(var.mean()):.3f} {var.attrs.get('units', '')}")


run("summary -h")

Usage: earthkit summary [OPTIONS] SOURCE

  Summarise the variables of SOURCE.

  SOURCE: The earthkit-data source to read, as [NAME:]VALUE, with NAME 'file' if not given. VALUE is
  passed to the source as its first argument, e.g. a file path (glob patterns allowed) or
  'url:https://myhost.int/file.nc', or as its request if it is a JSON object, e.g. 'mars:{"param":
  "2t", "levtype": "sfc"}'. The 'dataset' key of a JSON request, if any, is passed as the first
  argument, e.g. 'cds:{"dataset": "reanalysis-era5-single-levels", ...}'. Several sources are
  merged. '-' reads GRIB data from stdin.

Options:
  -p, --profile TEXT  Profile used when opening the data with Xarray: the name of an earthkit-data
                      built-in profile, e.g. 'earthkit', 'mars' or 'grib', or the path of a YAML
                      file holding a profile. Can be repeated, in which case the profiles are
                      layered in order, each overriding the options of the previous ones. Uses the

A source is given as `[NAME:]VALUE`, where `NAME` is an earthkit-data source, `file` if not given:

In [6]:
run("summary input.nc")

t2m: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 279.993 K
tp: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 0.498 m


The same command reads any other earthkit-data source, e.g. `url:https://myhost.int/file.nc`, or
`'mars:{"param": "2t", "levtype": "sfc"}'`, whose JSON value is passed to the source as its request.

A positional `SOURCE` takes one or more values, and several sources are merged into a single `multi` source.
`file` sources also accept glob patterns:

In [7]:
run("summary part1.nc part2.nc")
run("summary 'part*.nc'")

t2m: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 279.993 K
tp: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 0.498 m
t2m: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 279.993 K
tp: {'time': 48, 'latitude': 3, 'longitude': 4}, mean 0.498 m


The values are checked before the command runs, e.g. a `file` source must exist:

In [8]:
run("summary missing.nc")

Usage: earthkit summary [OPTIONS] SOURCE
Try 'earthkit summary --help' for help.

Error: Invalid value for 'SOURCE': Path 'missing.nc' does not exist.


## 3. Writing data

`target_options()` adds an earthkit-data target in the same way. The command receives a `Target`, and writes
data to it with `Target.to_target()`, which calls `earthkit.data.to_target()`.

This command also uses `split_csv` as the callback of a repeatable option, so that `--variables t2m,tp` and
`--variables t2m --variables tp` both give `["t2m", "tp"]`:

In [9]:
@earthkit.command(
    help=f"""Compute the time mean of the variables of SOURCE and write it to TARGET.

SOURCE: {SOURCE_HELP}

TARGET: {TARGET_HELP}
"""
)
@add_options([source_options(positional=True), target_options(positional=True)])
@click.option("--variables", multiple=True, callback=split_csv, help="Variables to keep. Comma-separated or repeated.")
def time_mean(source, target, variables):
    ds = source.to_xarray()
    if variables:
        ds = ds[variables]
    target.to_target(ds.mean("time", keep_attrs=True))


run("time-mean input.nc mean.nc --variables t2m")
run("summary mean.nc")

t2m: {'latitude': 3, 'longitude': 4}, mean 279.993 K


A target is also given as `[NAME:]VALUE`. A JSON `VALUE` is passed to the target as its keyword arguments, e.g.
to use the options of the `file` target, or to write to Zarr with
`'zarr:{"xarray_to_zarr_kwargs": {"store": "out.zarr"}}'`:

In [10]:
run("""time-mean input.nc 'file:{"file": "mean_all.nc"}'""")
run("summary mean_all.nc")

t2m: {'latitude': 3, 'longitude': 4}, mean 279.993 K
tp: {'latitude': 3, 'longitude': 4}, mean 0.498 m


## 4. Several sources

A command with several sources, or several targets, gives each of them a name, which is also the name of the
argument passed to the command and of its option. Here `source_options("source_1")` and
`source_options("source_2")` give the `--source-1` and `--source-2` options. Without `positional=True`, sources
and targets are required options, and only the default `source` and `target` have the short flags `-s` and
`-t`. As options have their own help, the docstring of the command does not need to describe them:

In [11]:
@earthkit.command()
@add_options([source_options("source_1"), source_options("source_2"), target_options()])
def difference(source_1, source_2, target):
    """Write the difference SOURCE_1 - SOURCE_2 to TARGET."""
    target.to_target(source_1.to_xarray() - source_2.to_xarray())


run("difference --source-1 part2.nc --source-2 mean.nc -t anomaly.nc")
run("summary anomaly.nc")

t2m: {'time': 24, 'latitude': 3, 'longitude': 4}, mean 0.023 K


A command can only have one positional source, as its `SOURCE` argument takes any number of values, so use
options for commands with several sources.

## 5. Selecting items with `--index`

`index_option` adds the shared `--index`/`-i` option, which selects items by position. It is passed to the
command as an integer, a list of integers or a slice, ready to index a list or an array, or `None` if not
given:

In [12]:
@earthkit.command()
@add_options([source_options(), index_option])
def times(source, index):
    """Print the times of SOURCE, or those selected by --index."""
    values = source.to_xarray().time.values
    if index is not None:
        values = np.atleast_1d(values[index])
    for value in values:
        click.echo(np.datetime_as_string(value, unit="h"))


run("times -s input.nc --index 0:3")
run("times -s input.nc --index 1,3,-1")

2020-01-01T00
2020-01-01T06
2020-01-01T12
2020-01-01T06
2020-01-01T18
2020-01-12T18


Invalid values are reported before the command runs:

In [13]:
run("times -s input.nc --index first")

Usage: earthkit times [OPTIONS]
Try 'earthkit times --help' for help.

Error: Invalid value for '-i' / '--index': Expected an integer, a comma-separated list or a START:STOP[:STEP] slice, got 'first'


## 6. Shipping the commands in a package

To ship commands with an earthkit package, put them in a module of the `earthkit.cli` namespace package, named
after the package, e.g. `earthkit/cli/mypackage.py` for earthkit-mypackage. The `earthkit.cli` directory has no
`__init__.py`, as it is shared by all earthkit packages:

```text
src/
└── earthkit/
    ├── cli/
    │   └── mypackage.py      # no __init__.py in earthkit/cli
    └── mypackage/
        ├── __init__.py
        └── ...
```

The module registers its commands when it is imported, which earthkit-utils does the first time the commands
are needed, e.g. for `earthkit -h`. Keep it fast to import: only import click, `earthkit.cli` and light standard
library modules at module level, and import anything else, including your own package, inside the commands:

```python
# src/earthkit/cli/mypackage.py
import click

from earthkit.cli.main import earthkit
from earthkit.cli.standard_args import SOURCE_HELP, TARGET_HELP, add_options, source_options, target_options


@earthkit.command(
    help=f"""Compute the time mean of the variables of SOURCE and write it to TARGET.

SOURCE: {SOURCE_HELP}

TARGET: {TARGET_HELP}

\b
Example:
    earthkit time-mean input.grib output.nc
"""
)
@add_options([source_options(positional=True), target_options(positional=True)])
def time_mean(source, target):
    import earthkit.mypackage as ekm  # imported only when the command runs

    target.to_target(ekm.time_mean(source.to_xarray()))
```

Include the namespace package in the distribution, and depend on earthkit-utils (which provides the `earthkit`
command and click) and on earthkit-data, e.g. with setuptools:

```toml
[project]
dependencies = ["earthkit-data", "earthkit-utils"]

[tool.setuptools.packages.find]
include = ["earthkit.mypackage", "earthkit.cli"]
where = ["src/"]
```

Once the package is installed, its commands are listed by `earthkit -h`, and `earthkit info` shows which
package provides them:

```console
$ earthkit info
earthkit-utils        1.1.0
earthkit-mypackage    0.1.0  (commands: time-mean)
```

A package with many commands can use a sub-package `earthkit/cli/mypackage/` instead, whose `__init__.py`
imports the modules defining the commands, see [Command line interface standards](../concepts/cli.rst).

### Testing the commands

The commands are tested like any click command, by invoking the shared group with `CliRunner`, as the `run`
helper of this tutorial does. Import the module of the commands so that they are registered:

```python
from click.testing import CliRunner

from earthkit.cli.main import earthkit
import earthkit.cli.mypackage  # noqa: F401, registers the commands


def test_time_mean(tmp_path):
    result = CliRunner().invoke(earthkit, ["time-mean", "input.nc", str(tmp_path / "out.nc")])
    assert result.exit_code == 0, result.output
```